# 2.2 RoPE：SIMD 与 SIMT

## 小节概述

目标：读懂 pair-planar 数据布局，编译并运行 910B3 SIMD Kernel，再说明 950 SIMT 模板的并行映射。

<img src="images/rope_pair_planar_flow.svg" width="760" style="display:block; margin-left:0;" />


## SIMD Kernel

`rope_simd_kernel.cpp` 先把 even/odd/cos/sin 搬到 LocalTensor，再用 `Mul/Sub/Add` 批量计算旋转结果。下面只保留解释主线所需的片段，完整源码位于 `src/rope_simd_kernel.cpp`。


```cpp
AscendC::DataCopy(xe, xEvenGm, TOTAL_PAIRS);
AscendC::DataCopy(xo, xOddGm, TOTAL_PAIRS);
AscendC::DataCopy(c,  cosGm,  TOTAL_PAIRS);
AscendC::DataCopy(s,  sinGm,  TOTAL_PAIRS);

AscendC::Mul(te1, xe, c, TOTAL_PAIRS);
AscendC::Mul(te2, xo, s, TOTAL_PAIRS);
AscendC::Sub(oe, te1, te2, TOTAL_PAIRS);
AscendC::Mul(to1, xe, s, TOTAL_PAIRS);
AscendC::Mul(to2, xo, c, TOTAL_PAIRS);
AscendC::Add(oo, to1, to2, TOTAL_PAIRS);
```

入口中的 `KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY)` 把任务限定为 Vector Core。`TOTAL_PAIRS=384` 对应 12 heads × 32 pairs；本实现是教学用单 tile、单缓冲，不预设为性能最优版本。


## RTC Host 与执行

Host 侧生命周期为：创建 RTC Program → 编译 → 读取二进制 → 加载 Vector Core Binary → 获取函数 → 追加 6 个设备指针 → 启动 → 同步 → 校验 → 逆序释放。

```cpp
aclrtcCompileProg(prog, numOptions, options);
aclrtBinaryLoadFromData(elfBin.data(), elfSize, &loadOption, &binHandle);
aclrtBinaryGetFunction(binHandle, "rope_simd", &funcHandle);
aclrtLaunchKernelWithConfig(funcHandle, blockDim, stream, nullptr, argsHandle, nullptr);
aclrtSynchronizeStream(stream);
```

下面直接编译 Host 程序并启动 Kernel；`run_rope_lab.sh` 仅保留为终端快捷入口。任一 ACL 错误都会记录调用点、进入统一释放段并返回非零。


In [ ]:
%%bash
set -e

# 使用当前 CANN 安装目录编译 RTC Host。
CANN_PREFIX="${ASCEND_HOME:-${ASCEND_HOME_PATH:-${ASCEND_TOOLKIT_HOME:?请设置 ASCEND_HOME}}}"
source "$CANN_PREFIX/set_env.sh"
mkdir -p build

g++ -std=c++17 \
  -I"$CANN_PREFIX/include" -I"$CANN_PREFIX/include/acl" \
  -L"$CANN_PREFIX/lib64" -Wl,-rpath,"$CANN_PREFIX/lib64" \
  src/rope_simd_rtc.cpp -lascendcl -lacl_rtc -o build/rope_simd_rtc

# Host 通过 RTC 编译、加载并启动 RoPE SIMD Kernel。
LD_LIBRARY_PATH="$CANN_PREFIX/lib64:${LD_LIBRARY_PATH:-}" \
  ./build/rope_simd_rtc --kernel src/rope_simd_kernel.cpp --warmup 2 --repeat 5


成功时最后一行以 `ROPE_RESULT status=PASS` 开头，并包含 `cases=4/4`、`max_error`、`tolerance`、`compile_seconds`、`device_mean_us`、`reference_mean_us`、`fallback=0`、`path=ASCENDC_SIMD_RTC` 和 `device_id=0`。

其中 `device_mean_us` 是 Host 计得的 Kernel launch + Stream synchronize 口径，不含 RTC 编译、H2D、D2H；它不是 profiler 的纯 Kernel 时间。任何编译、加载、执行或校验失败都应返回非零。


## SIMT 模板

`rope_simt_950.asc` 用 `blockIdx/threadIdx` 计算全局线程号，一个线程处理一个 pair，并用边界判断防止越界：


```cpp
unsigned int p = blockIdx.x * blockDim.x + threadIdx.x;
if (p >= count) {
    return;
}
out_even[p] = x_even[p] * cos[p] - x_odd[p] * sin[p];
out_odd[p]  = x_even[p] * sin[p] + x_odd[p] * cos[p];
```

模板只面向 Ascend 950（`dav-3510`、`--enable-simt`）。它与 SIMD 使用同一 pair-planar 数学合同，但当前课程没有 950 实机证据，因此状态必须保持 `DEFERRED_UNSUPPORTED_TARGET`。


## 课后实践

解释 SIMD 中“向量指令一次处理一段数据”和 SIMT 中“一个线程处理一个 pair”的区别，并说明：

1. 两条路径为什么必须保持相同输入输出布局；
2. `AIV_ONLY`、`fallback=0`、`device_id=0` 各自证明什么；
3. 为什么 910B3 的 PASS 不能推导出 950 SIMT PASS。


In [ ]:
# 完成练习后按需执行；Notebook 不会自动展开答案。
!cat answer/02.02_answer.md
